In [ ]:
import os
import numpy as np
from obspy import read
from scipy.signal import butter, filtfilt
import csv

np.seterr(invalid="ignore")


{'divide': 'warn', 'over': 'warn', 'under': 'ignore', 'invalid': 'ignore'}

In [ ]:
# Separates files by channel (only certain ones are accepted as others are different shaped waveforms)

def channel_to_component(channel):
    channel = channel.upper()

    allowed = {
        "HHE": "E",
        "HHN": "N",
        "HHZ": "Z",
        "BHE": "E",
        "BHN": "N",
        "BHZ": "Z",
        "CZ": "Z",
    }
    return allowed.get(channel)

In [ ]:
# Loads files from one station folder and returns [E, N, Z] array choronologically ordered

def load_stream_from_station(st_path):
    station_files = {"E": [], "N": [], "Z": []}
    for fname in os.listdir(st_path):
        if not fname.endswith(".mseed"):
            continue
        parts = fname.split(".")
        if len(parts) < 4:
            continue
        chan = parts[3].split("__")[0]
        comp = channel_to_component(chan)
        if comp is None:
            continue
        station_files[comp].append(os.path.join(st_path, fname))

    components = {}
    for comp in ["E", "N", "Z"]:
        if len(station_files[comp]) == 0:
            raise ValueError(f"Missing component {comp} in station")

        sorted_files = sorted(
            station_files[comp],
            key=lambda f: read(f)[0].stats.starttime
        )
        traces = [read(f)[0].data.astype(float) for f in sorted_files]
        components[comp] = np.concatenate(traces)

    min_len = min(len(components["E"]), len(components["N"]), len(components["Z"]))
    stream_array = np.stack([
        components["E"][:min_len],
        components["N"][:min_len],
        components["Z"][:min_len],
    ], axis=0)
    return stream_array

In [ ]:
# Zero-phase Butterworth bandpass filter

def bandpass(x, fs, fmin, fmax, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [fmin / nyq, fmax / nyq], btype="band")
    return filtfilt(b, a, x)

In [ ]:
# Custom STA/LTA function considering ishift and using absolute amplitude |x|

def compute_sta_lta(x, sta, lta, ishift):
    
    if lta <= sta:
        raise ValueError("LTA must be larger than STA")

    amp = np.abs(x.astype(float))

    sta_sum = np.convolve(amp, np.ones(sta), "valid")
    lta_sum = np.convolve(amp, np.ones(lta), "valid")

    # Apply shift so that LTA is strictly before STA
    shift = max(0, int(ishift))
    if shift > 0:
        lta_sum = lta_sum[:-shift]
        sta_sum = sta_sum[shift:]

    min_len = min(len(sta_sum), len(lta_sum))
    sta_sum = sta_sum[:min_len]
    lta_sum = lta_sum[:min_len]

    # Avoid division by zero
    lta_sum[lta_sum == 0] = 1e-12

    sta_mean = sta_sum / float(sta)
    lta_mean = lta_sum / float(lta)
    ratio = sta_mean / lta_mean

    # Pad on the left so that output has same length as x
    pad = len(x) - len(ratio)
    if pad > 0:
        ratio = np.pad(ratio, (pad, 0), mode="constant")

    return ratio

In [ ]:
# Computes coherence between three components using Pearson correlation over a sliding window

def compute_coherence(filtered, step, win_len):
    n_comp, n_samples = filtered.shape
    if n_comp != 3:
        raise ValueError("compute_coherence expects exactly 3 components (E, N, Z)")

    if win_len <= 1:
        return np.zeros(max(1, n_samples // step), dtype=float)

    # Number of Delta steps used elsewhere (e.g. ratios[:, ::step])
    n_steps = len(range(0, n_samples, step))
    coherence = np.zeros(n_steps, dtype=float)

    for k in range(n_steps):
        end_idx = k * step
        if end_idx <= 0:
            coherence[k] = 0.0
            continue

        start_idx = max(0, end_idx - win_len)
        seg = filtered[:, start_idx:end_idx]

        if seg.shape[1] < 2:
            coherence[k] = 0.0
            continue

        # Each row is a component; compute 3x3 correlation matrix
        corr = np.corrcoef(seg)
        # Replace NaNs (can occur for near-constant windows)
        corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)

        r_en = corr[0, 1]
        r_ez = corr[0, 2]
        r_nz = corr[1, 2]
        coherence[k] = np.mean(np.abs([r_en, r_ez, r_nz]))

    return coherence

In [ ]:
detectors = [
    {"Delta": 0.280, "L_wind": 4, "Ishift": 4,  "Sigma": 6,
     "Thresh_1": 2.55, "Thresh_2": 2.50, "Cohmin": 0.8,
     "NDMIN": 6,  "Freq": [2.0, 6.0]},
    {"Delta": 0.100, "L_wind": 4, "Ishift": 10, "Sigma": 6,
     "Thresh_1": 2.55, "Thresh_2": 2.50, "Cohmin": 0.8,
     "NDMIN": 15, "Freq": [5.0, 15.0]},
    {"Delta": 0.140, "L_wind": 4, "Ishift": 7,  "Sigma": 6,
     "Thresh_1": 2.50, "Thresh_2": 2.60, "Cohmin": 0.3,
     "NDMIN": 10, "Freq": [4.0, 10.0]},
    {"Delta": 0.050, "L_wind": 4, "Ishift": 20, "Sigma": 6,
     "Thresh_1": 2.50, "Thresh_2": 2.60, "Cohmin": 0.3,
     "NDMIN": 15, "Freq": [10.0, 35.0]},
]

In [ ]:
def replica_model_with_coherence(stream, detectors, fs=100.0):
    
    results = []

    for det in detectors:
        delta = det["Delta"]
        sta = int(det["L_wind"] * delta * fs)
        lta = int(det["Sigma"]   * delta * fs)
        ishift = int(det["Ishift"] * delta * fs)
        ndmin = det["NDMIN"]
        thr1 = det["Thresh_1"]
        thr2 = det["Thresh_2"]
        cohmin = det["Cohmin"]
        fmin, fmax = det["Freq"]

        # Band-pass filter all components
        filtered = np.array([
            bandpass(stream[i], fs, fmin, fmax)
            for i in range(3)
        ])  # shape: (3, n_samples)

        # STA/LTA ratio per component
        ratios = np.array([
            compute_sta_lta(filtered[i], sta, lta, ishift)
            for i in range(3)
        ])  # shape: (3, n_samples)

        # Downsample by Delta in samples
        step = max(1, int(delta * fs))
        ratios_ds = ratios[:, ::step]  # shape: (3, n_steps)

        # Coherence per Delta step using the same step and STA as window length
        coherence = compute_coherence(filtered, step=step, win_len=sta)

        # Align coherence length with downsampled ratios if needed
        n_steps = ratios_ds.shape[1]
        if len(coherence) > n_steps:
            coherence = coherence[:n_steps]
        elif len(coherence) < n_steps:
            coherence = np.pad(coherence, (0, n_steps - len(coherence)), mode="edge")

        # Use maximum STA/LTA ratio across components at each Delta step
        ratios_max = np.max(ratios_ds, axis=0)

        # Two trigger conditions (for detectors 0 and 1 coherence threshold doesnt mean anything as it is higher than regular threshold)
        trig_with_coh = (ratios_max > thr1) & (coherence >= cohmin)
        trig_plain    = (ratios_max > thr2)
        trigger = trig_with_coh | trig_plain

        # Enforce NDMIN in units of Delta steps
        #
        # records the detection at the *start* of the consecutive above-threshold run
        # (onset), not at the step where NDMIN is reached. This reduces the built-in delay of
        # roughly (NDMIN-1) * Delta seconds.
        detections = []
        count = 0
        for i, t in enumerate(trigger):
            if t:
                count += 1
                if count == ndmin:
                    onset_idx = i - ndmin + 1
                    onset_idx = max(0, onset_idx)
                    # Use STA/LTA strength at onset 
                    strength = ratios_max[onset_idx]
                    detections.append((onset_idx, strength))
            else:
                count = 0

        results.append(detections)

    return results


In [ ]:
# Clusters raw detections that are within a certain time window which is 3 seconds?

def cluster_detections(det_results, detectors, cluster_window=3.0):
   
    points = []

    for d_idx, det in enumerate(det_results):
        delta = detectors[d_idx]["Delta"]
        for idx, strength in det:
            t = idx * delta
            points.append((t, strength, d_idx))

    if not points:
        return []

    # Sort by time
    points.sort(key=lambda x: x[0])

    clusters = []
    current = [points[0]]

    for p in points[1:]:
        if p[0] - current[-1][0] < cluster_window:
            current.append(p)
        else:
            clusters.append(current)
            current = [p]

    clusters.append(current)
    return clusters


In [ ]:
# Extracts P picks from detectors 2 and 3 and S picks from detectors 0 and 1 (if there is no P or S pick, field left empty)

def extract_p_s(det_results, detectors, cluster_window=3.0):
    
    events = cluster_detections(det_results, detectors, cluster_window=cluster_window)
    picks = []

    # P-type = detector index 2, 3; S-type = detector index 0, 1
    p_detectors = (2, 3)
    s_detectors = (0, 1)

    for event in events:
        # event is a list of (time, strength, d_idx)
        p_candidates = [(t, s) for t, s, d in event if d in p_detectors]
        s_candidates = [(t, s) for t, s, d in event if d in s_detectors]

        if p_candidates:
            p_time, p_strength = min(p_candidates, key=lambda x: x[0])
            p_prob = min(1.0, p_strength / 5.0)
        else:
            p_time, p_prob = None, None

        if s_candidates:
            s_time, s_strength = min(s_candidates, key=lambda x: x[0])
            s_prob = min(1.0, s_strength / 5.0)
        else:
            s_time, s_prob = None, None

        picks.append((p_time, p_prob, s_time, s_prob))

    return picks


In [ ]:
# Runs the detection on each 3-component miniSEED window and writes a CSV with filename, p_arrival_time, s_arrival_time

def run_detection_to_full_csv(root, detectors, out_csv, fs=100.0):
    rows = []

    for ts_name in sorted(os.listdir(root)):
        ts_path = os.path.join(root, ts_name)
        if not os.path.isdir(ts_path):
            continue

        for station_dir in sorted(os.listdir(ts_path)):
            st_path = os.path.join(ts_path, station_dir)
            if not os.path.isdir(st_path):
                continue

            # Build stream
            try:
                stream = load_stream_from_station(st_path)
            except Exception:
                continue

            # Use one representative file for metadata/file_name
            station_mseed_files = [
                os.path.join(st_path, fname)
                for fname in os.listdir(st_path)
                if fname.endswith(".mseed")
            ]
            if not station_mseed_files:
                continue

            rep_fname = None
            for pref in ("HHN", "HHZ", "HHE"):
                for fpath in station_mseed_files:
                    if f".{pref}__" in os.path.basename(fpath):
                        rep_fname = fpath
                        break
                if rep_fname is not None:
                    break
            if rep_fname is None:
                rep_fname = station_mseed_files[0]

            st_rep = read(rep_fname)[0]
            net = st_rep.stats.network
            sta = st_rep.stats.station
            starttime = st_rep.stats.starttime
            endtime = st_rep.stats.endtime

            # Run detector on this window
            det_results = replica_model_with_coherence(stream, detectors, fs=fs)
            picks = extract_p_s(det_results, detectors, cluster_window=3.0)

            # Build CSV rows, one per (P,S) pair
            for p_time, p_prob, s_time, s_prob in picks:
                # Leave missing phase arrivals empty
                p_abs = (starttime + p_time).datetime.replace(tzinfo=None) if p_time is not None else ""
                s_abs = (starttime + s_time).datetime.replace(tzinfo=None) if s_time is not None else ""
                start_dt = starttime.datetime.replace(tzinfo=None, microsecond=0)
                end_dt = endtime.datetime.replace(tzinfo=None, microsecond=0)

                # file_name column: relative path like "STATION/NET.STA..HHN__...mseed"
                file_name = f"{station_dir}/{os.path.basename(rep_fname)}"

                rows.append([
                    file_name,
                    str(p_abs) if p_abs != "" else "",
                    str(s_abs) if s_abs != "" else "",
                ])

    # Write CSV
    with open(out_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["file_name", "p_arrival_time", "s_arrival_time"])
        writer.writerows(rows)

    return out_csv


In [ ]:
root = "waveforms_earthquakes_nonoise"
out_path = "sta_lta_eq_nonoise_results.csv"
run_detection_to_full_csv(root, detectors, out_path, fs=100.0)
print("Wrote:", out_path)

Wrote: sta_lta_eq_nonoise_results_Z.csv
